# NMF on ModulAir 00686

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00068
data = pd.read_csv('MOD-00681-2-26data.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-29T13:41:45Z,575936314,2025-12-29T08:41:45Z,MOD-00681,94.9,6.1,0.0,0.0,0.0,0.0,0.0,...,10.497,38.829,NaN,NaN,NaN,14468.0,14493.0,14543.0,14518.0,2.71
2025-12-29T13:40:49Z,575936312,2025-12-29T08:40:49Z,MOD-00681,94.9,6.1,0.0,0.0,0.0,0.0,0.0,...,10.500,39.246,NaN,NaN,NaN,14468.0,14493.0,14543.0,14518.0,3.02
2025-12-29T13:39:47Z,575936317,2025-12-29T08:39:47Z,MOD-00681,94.9,6.1,0.0,0.0,0.0,0.0,0.0,...,10.500,39.246,NaN,NaN,NaN,14468.0,14493.0,14543.0,14518.0,2.92
2025-12-29T13:38:45Z,575936316,2025-12-29T08:38:45Z,MOD-00681,94.9,6.1,0.0,0.0,0.0,0.0,0.0,...,10.500,39.246,NaN,NaN,NaN,14468.0,14493.0,14543.0,14518.0,2.49
2025-12-29T13:37:48Z,575936310,2025-12-29T08:37:48Z,MOD-00681,94.9,6.1,0.0,0.0,0.0,0.0,0.0,...,10.500,39.663,NaN,NaN,NaN,14468.0,14493.0,14543.0,14518.0,1.45


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-29T13:41:45Z,2025-12-29T08:41:45Z,1112.629,4.079,10.497,38.829,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-29T13:40:49Z,2025-12-29T08:40:49Z,1128.639,4.044,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-29T13:39:47Z,2025-12-29T08:39:47Z,1119.608,4.079,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-29T13:38:45Z,2025-12-29T08:38:45Z,1114.682,3.813,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-29T13:37:48Z,2025-12-29T08:37:48Z,1119.197,3.811,10.500,39.663,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29T08:41:45Z,1112.629,4.079,10.497,38.829,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-12-29T08:40:49Z,1128.639,4.044,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-12-29T08:39:47Z,1119.608,4.079,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-12-29T08:38:45Z,1114.682,3.813,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-12-29T08:37:48Z,1119.197,3.811,10.500,39.663,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 08:41:45,1112.629,4.079,10.497,38.829,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-12-29 08:40:49,1128.639,4.044,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-12-29 08:39:47,1119.608,4.079,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-12-29 08:38:45,1114.682,3.813,10.500,39.246,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-12-29 08:37:48,1119.197,3.811,10.500,39.663,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,1085.50,26.84,40.33,1.99,28.99,2.61,1.15,0.41,0.59,0.49
1,2025-03-31 21:00:00,1054.31,23.56,40.84,2.76,34.32,4.57,1.52,0.48,0.66,0.59
2,2025-03-31 22:00:00,927.45,15.23,45.55,2.88,31.95,6.64,1.74,0.45,0.56,0.45
3,2025-03-31 23:00:00,742.65,8.55,61.71,2.79,15.64,2.74,0.75,0.19,0.22,0.15
4,2025-04-01 00:00:00,691.18,5.73,64.16,2.74,16.64,2.42,0.67,0.17,0.21,0.13
...,...,...,...,...,...,...,...,...,...,...,...
6421,2025-12-27 21:00:00,784.75,16.94,41.43,2.51,11.91,0.87,0.27,0.06,0.05,0.03
6422,2025-12-27 22:00:00,719.59,14.51,43.38,2.08,11.92,0.93,0.25,0.05,0.05,0.03
6423,2025-12-27 23:00:00,728.66,15.19,42.19,2.28,14.72,1.09,0.29,0.05,0.05,0.03
6424,2025-12-28 00:00:00,770.65,15.66,40.75,2.90,17.64,1.26,0.33,0.07,0.06,0.03


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,1085.50,26.84,40.33,1.99,28.99,2.61,1.15,0.41,0.59,0.49
2025-03-31 21:00:00,1054.31,23.56,40.84,2.76,34.32,4.57,1.52,0.48,0.66,0.59
2025-03-31 22:00:00,927.45,15.23,45.55,2.88,31.95,6.64,1.74,0.45,0.56,0.45
2025-03-31 23:00:00,742.65,8.55,61.71,2.79,15.64,2.74,0.75,0.19,0.22,0.15
2025-04-01 00:00:00,691.18,5.73,64.16,2.74,16.64,2.42,0.67,0.17,0.21,0.13


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.499204,0.656074,0.470705,0.000507,0.115365,0.081614,0.067134,0.079612,0.132287,0.357664
2025-03-31 21:00:00,0.484861,0.575898,0.476657,0.000703,0.136575,0.142902,0.088733,0.093204,0.147982,0.430657
2025-03-31 22:00:00,0.426520,0.372281,0.531629,0.000734,0.127144,0.207630,0.101576,0.087379,0.125561,0.328467
2025-03-31 23:00:00,0.341533,0.208995,0.720238,0.000711,0.062239,0.085679,0.043783,0.036893,0.049327,0.109489
2025-04-01 00:00:00,0.317863,0.140064,0.748833,0.000698,0.066218,0.075672,0.039113,0.033010,0.047085,0.094891


In [11]:
data.to_csv('MOD-000681_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.442241,0.672602,0.500548,0.002466,0.088759,0.189395,0.132453,0.107179,0.121317,0.231609
2025-03-31 21:00:00,0.442580,0.590130,0.500284,0.002327,0.101449,0.240099,0.169203,0.137318,0.152226,0.284178
2025-03-31 22:00:00,0.421545,0.376011,0.536060,0.002147,0.109534,0.227627,0.155833,0.125058,0.136663,0.253428
2025-03-31 23:00:00,0.389409,0.199999,0.698523,0.002622,0.032397,0.074206,0.051792,0.041878,0.051306,0.103751
2025-04-01 00:00:00,0.380425,0.128232,0.720353,0.002612,0.026403,0.064160,0.044842,0.036277,0.045101,0.091949
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 20:00:00,0.343033,0.325195,0.507377,0.002067,0.070083,0.033252,0.010356,0.004383,0.008117,0.027030
2025-12-27 21:00:00,0.345153,0.416729,0.490464,0.002151,0.059533,0.028186,0.009578,0.004611,0.009782,0.031478
2025-12-27 22:00:00,0.332960,0.353924,0.505102,0.002136,0.049276,0.024915,0.009187,0.004882,0.010203,0.031397


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.052060,0.085512,0.099961,0.049997
1,0.051292,0.073015,0.128371,0.059967
2,0.054611,0.041815,0.115849,0.068972
3,0.076346,0.017071,0.039033,0.019863
4,0.078989,0.006683,0.033827,0.017013
...,...,...,...,...
5879,0.053346,0.037477,0.001067,0.042906
5880,0.052102,0.051065,0.001927,0.034252
5881,0.054168,0.042082,0.002624,0.028372
5882,0.052083,0.044411,0.002656,0.035444


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.277922,0.922249,8.976514,0.032293,0.000010,0.000000,0.000000,0.000000,0.087688,0.305007
Feature 2,1.433682,7.010144,0.000000,0.009181,0.169529,0.000000,0.000000,0.000000,0.059956,0.231257
Feature 3,0.260789,0.097943,0.000000,0.000000,0.000000,1.526049,1.219497,1.033985,1.116686,1.960315
Feature 4,1.417398,0.306954,0.664646,0.000000,1.485313,0.737041,0.211027,0.076435,0.000000,0.000000


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.052060,0.085512,0.099961,0.049997
2025-03-31 21:00:00,0.051292,0.073015,0.128371,0.059967
2025-03-31 22:00:00,0.054611,0.041815,0.115849,0.068972
2025-03-31 23:00:00,0.076346,0.017071,0.039033,0.019863
2025-04-01 00:00:00,0.078989,0.006683,0.033827,0.017013
...,...,...,...,...
2025-12-27 20:00:00,0.053346,0.037477,0.001067,0.042906
2025-12-27 21:00:00,0.052102,0.051065,0.001927,0.034252
2025-12-27 22:00:00,0.054168,0.042082,0.002624,0.028372


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.277922,0.922249,8.976514,0.032293,0.000010,0.000000,0.000000,0.000000,0.087688,0.305007
Factor 2,1.433682,7.010144,0.000000,0.009181,0.169529,0.000000,0.000000,0.000000,0.059956,0.231257
Factor 3,0.260789,0.097943,0.000000,0.000000,0.000000,1.526049,1.219497,1.033985,1.116686,1.960315
Factor 4,1.417398,0.306954,0.664646,0.000000,1.485313,0.737041,0.211027,0.076435,0.000000,0.000000


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.700187,0.124546,0.010521,0.155778,0.008968
no,0.832701,0.125651,0.000000,0.000000,0.041647
no2,0.189708,0.765346,0.004966,0.042398,-0.002417
o3,0.955207,0.000000,0.000000,0.047491,-0.002698
bin0,0.000010,0.084696,0.000000,0.938811,-0.023517
bin1,0.000000,0.000000,0.482869,0.635336,-0.118206
bin2,0.000000,0.000000,0.757314,0.357012,-0.114326
bin3,0.000000,0.000000,0.843892,0.169947,-0.013839
bin4,0.222292,0.080669,0.697739,0.000000,-0.000700
bin5,0.336554,0.135436,0.533153,0.000000,-0.005142


In [20]:
results.to_csv('681_factor_results.csv')
comp.to_csv('681_factor_comp.csv')
res.to_csv('681_factor_resid.csv')